# Data-Efficient and Uncertainty-Calibrated Cross-Site Solar PV Power Forecasting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aminur167/cross-site-solar-forecasting/blob/main/notebooks/01_cross_site_solar_forecasting_demo.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-Repository-blue.svg)](https://github.com/aminur167/cross-site-solar-forecasting)

**Author:** Aminur Rahman (HSTU, Bangladesh)  
**Target Journal:** *Energy and AI* (Elsevier)  

This interactive notebook demonstrates the complete end-to-end pipeline:
1. Data engineering & physical constraint checking
2. Baseline benchmarking (Persistence vs. Ridge vs. GBDT)
3. Cross-site transfer and few-shot adaptation frontier ($k \in \{0, 1, 3, 7, 14\}$ days)
4. Conformalized Quantile Regression (CQR) with 90% coverage guarantees
5. Diebold-Mariano hypothesis testing

In [ ]:
# 1. Setup & Imports
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

print(f"NumPy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"LightGBM Version: {lgb.__version__}")

In [ ]:
# 2. Load Processed Datasets (Source: Site A, Target: Site B)
data_path_a = "../data/processed/site_a_features.csv"
data_path_b = "../data/processed/site_b_features.csv"

df_a = pd.read_csv(data_path_a, index_col=0, parse_dates=True)
df_b = pd.read_csv(data_path_b, index_col=0, parse_dates=True)

print(f"Site A (Source) feature matrix: {df_a.shape}")
print(f"Site B (Target) feature matrix: {df_b.shape}")
df_a[['active_power_kw', 'capacity_factor', 'temp_c', 'ghi_wm2']].head()

In [ ]:
# 3. Diurnal Solar Generation Profiles
plt.figure(figsize=(10, 4), dpi=150)
sample_day = df_a.loc['2025-06-01':'2025-06-05']
plt.plot(sample_day.index, sample_day['active_power_kw'], color='#2b5c8f', label='Measured Inverter Active Power (kW)')
plt.ylabel('Power (kW)')
plt.title('Representative 5-Day Solar Generation Telemetry (Site A, 50 kWp)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 4. Few-Shot Target Adaptation Experiment (Summary Results)
results_path = "../experiments/results/cross_site_adaptation_results.json"
import json
with open(results_path) as f:
    res = json.load(f)

budgets = [1, 3, 7, 14]
to_mae = [res["Zero_Shot_and_Adaptation"][f"k_{k}_days"]["target_only_metrics"]["nMAE_pct"] for k in budgets]
tl_mae = [res["Zero_Shot_and_Adaptation"][f"k_{k}_days"]["transfer_adapted_metrics"]["nMAE_pct"] for k in budgets]

plt.figure(figsize=(8, 4.5), dpi=150)
plt.plot(budgets, to_mae, 'o--', color='#d95f02', label='Target-Only (Train from scratch)')
plt.plot(budgets, tl_mae, 's-', color='#2b5c8f', label='Transfer-Adapted (Ours)')
plt.xlabel('Target Adaptation Budget (Days)')
plt.ylabel('nMAE (%)')
plt.title('Data-Efficient Adaptation Frontier (Site A -> Site B)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 5. Conformalized Quantile Regression Calibration Verification
uncert_path = "../experiments/results/uncertainty_robustness_results.json"
with open(uncert_path) as f:
    u_res = json.load(f)

cal_data = u_res["Uncertainty_Calibration_Regimes"]
print("Conformal Uncertainty Calibration Summary (Nominal 90% Interval):")
for k, v in cal_data.items():
    print(f"- {k:<30}: Empirical Coverage = {v['PICP_pct']:.2f}% | MPIW = {v['MPIW']:.4f}")